In [1]:
import pandas as pd
from datetime import datetime
import time
import numpy as np
import matplotlib.pyplot as plot
import scipy.stats

from scipy.interpolate import interp1d, CubicSpline
from scipy.stats import norm
from scipy.integrate import quad
from scipy.optimize import minimize, least_squares, newton
import pandas_datareader.data as web
import yfinance as yf
from nelson_siegel_svensson import NelsonSiegelSvenssonCurve
from nelson_siegel_svensson.calibrate import calibrate_nss_ols
from joblib import Parallel, delayed
import gc

Loading the Data

In [20]:
# Loading the .dta files

path = "C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\option_prices2003\option_prices2003.dta"
data_full = pd.read_stata(path)
print(data_full.info())


<class 'pandas.core.frame.DataFrame'>
Int64Index: 30798708 entries, 0 to 30798707
Data columns (total 34 columns):
 #   Column           Dtype         
---  ------           -----         
 0   secid            float64       
 1   date             datetime64[ns]
 2   symbol           object        
 3   symbol_flag      object        
 4   exdate           datetime64[ns]
 5   last_date        datetime64[ns]
 6   cp_flag          object        
 7   strike_price     float64       
 8   best_bid         float64       
 9   best_offer       float64       
 10  volume           float64       
 11  open_interest    float64       
 12  impl_volatility  float64       
 13  delta            float64       
 14  gamma            float64       
 15  vega             float64       
 16  theta            float64       
 17  optionid         float64       
 18  cfadj            float64       
 19  ss_flag          object        
 20  root             object        
 21  suffix           object      

In [ ]:
# Cusips to search
data_full = data_full[data_full["cusip"].isin(["64881510", "12486C10", "12490H10"])]
len(data_full)

211980

In [22]:
data_full[data_full["volume"] > 0]["volume"]
data_full[data_full["volume"] > 0]["cusip"].value_counts()
data_full["cusip"].value_counts()


12490H10    114697
12486C10     83000
12483W10     14283
Name: cusip, dtype: int64

In [23]:
# Data pre processing and skimming to get the more liquid options

cols = ["date", "exdate", "cp_flag", "strike_price", "best_bid", "best_offer", "volume", "open_interest", "impl_volatility", "delta", "gamma", "theta", "vega", "optionid", "cusip", "issuer", "exercise_style"]
data = data_full[cols]
data = data.rename(columns={"delta":"ddelta", "gamma":"dgamma", "theta":"dtheta", "vega":"dvega"})
data["cusip"] = data["cusip"].astype(str)
data["optionid"] = data["optionid"].astype(str)

data = data[data["exercise_style"] == "E"]

data = data[data["impl_volatility"] > 0]
data = data[data["best_bid"] > 0]
data = data[data["best_bid"] < data["best_offer"]]

data["days_to_mat"] = ((data["exdate"] - data["date"]).dt.total_seconds()/(24*3600))*5/7
data["time_to_mat"] = data["days_to_mat"]/252
data["strike_price"] = data["strike_price"]/1000
data["cp_flag_one"] = data["cp_flag"].apply(lambda x: 1 if x == "C" else(-1 if x == "P" else 0))


len(data["cusip"])

159526

In [24]:
del data_full
gc.collect()

22

In [25]:
# Loading the underlying Realized Volatility data

path_real_vol = "C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\historical_vol199601_201408\historical_vol199601_201408.csv"
data_real_vol = pd.read_csv(path_real_vol)
print(data_real_vol.info())

C:\Users\chril\AppData\Local\Temp\ipykernel_2400\3227490044.py:4: DtypeWarning: Columns (4,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  data_real_vol = pd.read_csv(path_real_vol) #, sep = ",")#, encoding="ANSI")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175664114 entries, 0 to 175664113
Data columns (total 12 columns):
 #   Column          Dtype  
---  ------          -----  
 0   secid           int64  
 1   date            object 
 2   days            int64  
 3   volatility      float64
 4   cusip           object 
 5   ticker          object 
 6   sic             float64
 7   index_flag      int64  
 8   exchange_d      int64  
 9   class           object 
 10  issue_type      object 
 11  industry_group  float64
dtypes: float64(3), int64(4), object(5)
memory usage: 15.7+ GB
None


In [26]:
# Selecting Option data for the Volatility surfaces calibrations

cus = data["cusip"].value_counts().head(30)
cus = data_real_vol[data_real_vol["cusip"].isin(cus.index)]["cusip"].value_counts().head(10)
columns = ["date", "days", "volatility", "cusip"]
data_real_vol = data_real_vol[data_real_vol["cusip"].isin(cus.index)][columns].copy()

data = data[data["cusip"].isin(cus.index)].copy()

len(data_real_vol["cusip"].value_counts())
len(data["cusip"].value_counts())
data["cusip"].value_counts()
len(data["cusip"])



159526

In [27]:
# Loading the zero coupon yield curve
path3 = "C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\zero_coupon_yield_curve199601_201408\zero_coupon_yield_curve199601_201408.dta"
data3 = pd.read_stata(path3)
data3.dtypes


date    datetime64[ns]
days           float64
rate           float64
dtype: object

In [ ]:
### Here I create the interpolated function for all points of the rf curve for every respective date 
interpolated = list()
for i in pd.unique(data3["date"]):
    data3f = data3.loc[(data3["date"] == i)]
    interpolated.append([i, interp1d(data3f["days"], data3f["rate"], kind = 'quadratic', fill_value = "extrapolate")])




interpolated = pd.DataFrame(interpolated)
interpolated.set_index(0, inplace = True)
data = data.merge(interpolated, how='left', left_on="date", right_on=interpolated.index)
print(data[data[1].isna()]["date"].value_counts())
data = data.rename(columns={1:"funct"})
data = data.dropna(subset=["funct"])

data["rf"] = data.apply(lambda row: row["funct"](row["days_to_mat"])/100, axis=1)
data = data.drop("funct", axis=1)


data
len(data["cusip"])

2003-10-13    708
2003-11-11    689
Name: date, dtype: int64


158129

In [29]:
# Load the respective option underlying (stock) data

path_stock = "C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\security_prices199601_201408\security_prices199601_201408.dta"
data_stock = pd.read_stata(path_stock)
data_stock["cusip"] = data_stock["cusip"].astype(str)
data_stock[data_stock["close"] < 0]["close"] = data_stock[data_stock["close"] < 0]["close"]*(-1)

C:\Users\chril\AppData\Local\Temp\ipykernel_2400\2192684042.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_stock[data_stock["close"] < 0]["close"] = data_stock[data_stock["close"] < 0]["close"]*(-1) # All close prices are positive


In [ ]:
# Merge the option data, with the underlying's data

cols = ["date", "cusip", "close", "cfadj", "cfret"]
data = data.merge(data_stock[cols], left_on = ('cusip', 'date'), right_on = ('cusip', 'date'), how = 'left')

data["price"] = data[["best_bid","best_offer"]].mean(axis=1)
data["stock_price"] = data["close"]*data["cfret"]
data["forward_price"] = data["stock_price"]*np.exp(data["rf"]*data["days_to_mat"]/252)
data

,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,impl_volatility,ddelta,...,days_to_mat,time_to_mat,cp_flag_one,rf,close,cfadj,cfret,price,stock_price,forward_price
0,2003-01-02,2003-01-18,C,210.0,0.65,1.05,0.0,174.0,0.219909,0.183096,...,11.428571,0.045351,1,0.013689,201.78,1.0,1.0,0.85,201.78,201.905307
1,2003-01-02,2003-01-18,C,170.0,31.20,32.60,0.0,0.0,0.428464,0.981064,...,11.428571,0.045351,1,0.013689,201.78,1.0,1.0,31.90,201.78,201.905307
2,2003-01-02,2003-01-18,C,180.0,21.40,22.80,0.0,0.0,0.363794,0.949103,...,11.428571,0.045351,1,0.013689,201.78,1.0,1.0,22.10,201.78,201.905307
3,2003-01-02,2003-01-18,C,190.0,12.10,13.00,0.0,1.0,0.282212,0.867788,...,11.428571,0.045351,1,0.013689,201.78,1.0,1.0,12.55,201.78,201.905307
4,2003-01-02,2003-01-18,C,200.0,4.30,5.00,3.0,150.0,0.234655,0.585990,...,11.428571,0.045351,1,0.013689,201.78,1.0,1.0,4.65,201.78,201.905307
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158124,2003-12-31,2004-09-18,P,1600.0,181.50,187.80,0.0,0.0,0.222895,-0.626782,...,187.142857,0.742630,-1,0.012519,1467.92,1.0,1.0,184.65,1467.92,1481.630797
158125,2003-12-31,2004-09-18,P,1650.0,215.70,221.80,0.0,0.0,0.218758,-0.691098,...,187.142857,0.742630,-1,0.012519,1467.92,1.0,1.0,218.75,1467.92,1481.630797
158126,2003-12-31,2004-09-18,P,1700.0,253.00,258.70,0.0,0.0,0.214971,-0.750037,...,187.142857,0.742630,-1,0.012519,1467.92,1.0,1.0,255.85,1467.92,1481.630797
158127,2003-12-31,2004-09-18,P,1750.0,292.60,298.20,0.0,0.0,0.210894,-0.803101,...,187.142857,0.742630,-1,0.012519,1467.92,1.0,1.0,295.40,1467.92,1481.630797


In [ ]:
# Black Scholes reverse implied volatility estimation:

def BS_pricer(S0, K, rf, impl_vol, tau, corp):
    d1 = 1/(impl_vol*np.sqrt(tau))*(np.log(S0/K) + (rf + (impl_vol**2)/2)*tau)
    d2 = d1 - impl_vol*np.sqrt(tau)
    BS_price = corp*norm.cdf(corp*d1)*S0 - corp*norm.cdf(corp*d2)*K*np.exp(-rf*tau)
    return(BS_price)


def BS_impl_vol(S0, K, rf, P, tau, corp):
    def obj_funct(impl_vol_guess):
        return((P - BS_pricer(S0, K, rf, impl_vol_guess, tau, corp))**2)
    
    impl_vol_guess = 0.3
    impl_vol_opt = newton(obj_funct, impl_vol_guess, tol = 1e-30, maxiter=200, disp = False)

    return(impl_vol_opt)

b=[]
for i in data.index:
    b.append(BS_impl_vol(data["stock_price"][i], data["strike_price"][i], data["rf"][i], data["price"][i], data["time_to_mat"][i], data["cp_flag_one"][i]) if 0 < BS_impl_vol(data["stock_price"][i], data["strike_price"][i], data["rf"][i], data["price"][i], data["time_to_mat"][i], data["cp_flag_one"][i]) < 10 else 0)
data["BS_impl_vol"] = b

In [ ]:
# Defining and running parallel realized volatility points interpolation and using functions to estimate relative option realized volatility

def real_vol_funct(data_real_vol_f):
    abc = interp1d(data_real_vol_f["days"], data_real_vol_f["volatility"], kind = 'linear', fill_value = "extrapolate")
    d = pd.DataFrame([[data_real_vol_f["date"].iloc[0], data_real_vol_f["cusip"].iloc[0], [abc]]])
    return d




grouped_data = data_real_vol.groupby(['date', 'cusip'])
n_jobs = 12

result_real_vol = Parallel(n_jobs=n_jobs)(delayed(real_vol_funct)(group) for _, group in grouped_data) 

In [32]:
interpol = pd.DataFrame()
for i in range(0, len(grouped_data)):
    interpol = pd.concat([interpol, result_real_vol[i]], axis=0)
interpol

,0,1,2
0,1996-01-04,12483W10,[<scipy.interpolate._interpolate.interp1d obje...
0,1996-01-04,12490H10,[<scipy.interpolate._interpolate.interp1d obje...
0,1996-01-05,12483W10,[<scipy.interpolate._interpolate.interp1d obje...
0,1996-01-05,12490H10,[<scipy.interpolate._interpolate.interp1d obje...
0,1996-01-08,12483W10,[<scipy.interpolate._interpolate.interp1d obje...
...,...,...,...
0,2014-08-27,12490H10,[<scipy.interpolate._interpolate.interp1d obje...
0,2014-08-28,12486C10,[<scipy.interpolate._interpolate.interp1d obje...
0,2014-08-28,12490H10,[<scipy.interpolate._interpolate.interp1d obje...
0,2014-08-29,12486C10,[<scipy.interpolate._interpolate.interp1d obje...


In [ ]:
interpol = pd.DataFrame(interpol)
cols_inter = ["date", "cusip", "funct"]
interpol.columns = cols_inter
interpol["date"] = pd.to_datetime(interpol["date"])
data = data.merge(interpol, how='left', left_on=("date", "cusip"), right_on=("date", "cusip"))

data = data.dropna(subset=["funct"])



data["realized_vol_annualized"] = data.apply(lambda row: float(row["funct"][0](row["days_to_mat"])), axis=1)
data["realized_vol"] = data.apply(lambda row: float(row["funct"][0](row["days_to_mat"])*row["days_to_mat"]/252), axis=1)
data = data.drop("funct", axis=1)
data

,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,impl_volatility,ddelta,...,cp_flag_one,rf,close,cfadj,cfret,price,stock_price,forward_price,realized_vol_annualized,realized_vol
0,2003-01-02,2003-01-18,C,210.0,0.65,1.05,0.0,174.0,0.219909,0.183096,...,1,0.013689,201.78,1.0,1.0,0.85,201.78,201.905307,0.191958,0.008706
1,2003-01-02,2003-01-18,C,170.0,31.20,32.60,0.0,0.0,0.428464,0.981064,...,1,0.013689,201.78,1.0,1.0,31.90,201.78,201.905307,0.191958,0.008706
2,2003-01-02,2003-01-18,C,180.0,21.40,22.80,0.0,0.0,0.363794,0.949103,...,1,0.013689,201.78,1.0,1.0,22.10,201.78,201.905307,0.191958,0.008706
3,2003-01-02,2003-01-18,C,190.0,12.10,13.00,0.0,1.0,0.282212,0.867788,...,1,0.013689,201.78,1.0,1.0,12.55,201.78,201.905307,0.191958,0.008706
4,2003-01-02,2003-01-18,C,200.0,4.30,5.00,3.0,150.0,0.234655,0.585990,...,1,0.013689,201.78,1.0,1.0,4.65,201.78,201.905307,0.191958,0.008706
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158124,2003-12-31,2004-09-18,P,1600.0,181.50,187.80,0.0,0.0,0.222895,-0.626782,...,-1,0.012519,1467.92,1.0,1.0,184.65,1467.92,1481.630797,0.221764,0.164689
158125,2003-12-31,2004-09-18,P,1650.0,215.70,221.80,0.0,0.0,0.218758,-0.691098,...,-1,0.012519,1467.92,1.0,1.0,218.75,1467.92,1481.630797,0.221764,0.164689
158126,2003-12-31,2004-09-18,P,1700.0,253.00,258.70,0.0,0.0,0.214971,-0.750037,...,-1,0.012519,1467.92,1.0,1.0,255.85,1467.92,1481.630797,0.221764,0.164689
158127,2003-12-31,2004-09-18,P,1750.0,292.60,298.20,0.0,0.0,0.210894,-0.803101,...,-1,0.012519,1467.92,1.0,1.0,295.40,1467.92,1481.630797,0.221764,0.164689


In [ ]:
# Calculated the variance risk premium for each day, take the mean and assign it to every option on that day (market wide VRP)
data['VRP'] = (data['impl_volatility'] - data['realized_vol_annualized']).groupby(data['date'].dt.date).transform('mean')
data["moneyness"] = np.log(data["strike_price"]/data["stock_price"])
data

,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,impl_volatility,ddelta,...,rf,close,cfadj,cfret,price,stock_price,forward_price,realized_vol_annualized,realized_vol,VRP
0,2003-01-02,2003-01-18,C,210.0,0.65,1.05,0.0,174.0,0.219909,0.183096,...,0.013689,201.78,1.0,1.0,0.85,201.78,201.905307,0.191958,0.008706,0.034379
1,2003-01-02,2003-01-18,C,170.0,31.20,32.60,0.0,0.0,0.428464,0.981064,...,0.013689,201.78,1.0,1.0,31.90,201.78,201.905307,0.191958,0.008706,0.034379
2,2003-01-02,2003-01-18,C,180.0,21.40,22.80,0.0,0.0,0.363794,0.949103,...,0.013689,201.78,1.0,1.0,22.10,201.78,201.905307,0.191958,0.008706,0.034379
3,2003-01-02,2003-01-18,C,190.0,12.10,13.00,0.0,1.0,0.282212,0.867788,...,0.013689,201.78,1.0,1.0,12.55,201.78,201.905307,0.191958,0.008706,0.034379
4,2003-01-02,2003-01-18,C,200.0,4.30,5.00,3.0,150.0,0.234655,0.585990,...,0.013689,201.78,1.0,1.0,4.65,201.78,201.905307,0.191958,0.008706,0.034379
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158124,2003-12-31,2004-09-18,P,1600.0,181.50,187.80,0.0,0.0,0.222895,-0.626782,...,0.012519,1467.92,1.0,1.0,184.65,1467.92,1481.630797,0.221764,0.164689,0.085306
158125,2003-12-31,2004-09-18,P,1650.0,215.70,221.80,0.0,0.0,0.218758,-0.691098,...,0.012519,1467.92,1.0,1.0,218.75,1467.92,1481.630797,0.221764,0.164689,0.085306
158126,2003-12-31,2004-09-18,P,1700.0,253.00,258.70,0.0,0.0,0.214971,-0.750037,...,0.012519,1467.92,1.0,1.0,255.85,1467.92,1481.630797,0.221764,0.164689,0.085306
158127,2003-12-31,2004-09-18,P,1750.0,292.60,298.20,0.0,0.0,0.210894,-0.803101,...,0.012519,1467.92,1.0,1.0,295.40,1467.92,1481.630797,0.221764,0.164689,0.085306


In [ ]:
# 1. Minimum time to expiration — exclude options with < 6 trading days (near-expiry noise)
data = data[data["days_to_mat"] >= 5]

data = data[data["days_to_mat"] <= 252]

data = data[data["best_bid"] >= 0.05]

data = data[(data["best_offer"] - data["best_bid"]) / ((data["best_bid"] + data["best_offer"]) / 2) <= 0.5]




len(data)
len(data["optionid"].unique())
data["optionid"].value_counts().mean()

In [ ]:
# Export data
outputpath = "C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\data_output_final\data_output_2003_top3_qual.dta"
data.to_stata(outputpath)